In [ ]:
pip install scikeras


In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

import pickle


In [ ]:
df = pd.read_csv('/content/Churn_Modelling.csv')
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [ ]:
# There is no NULL Value, so we'll directly dropm the unnecessary cols
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)


In [ ]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [ ]:
# remaining task at one place

label_encoder_gender = LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown= 'ignore')
encoding_geo = onehot_encoder_geo.fit_transform(df[['Geography']]).toarray()
encoded_geo_df = pd.DataFrame(encoding_geo, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

df = pd.concat([df.drop('Geography', axis=1), encoded_geo_df], axis=1)

# feature split
X = df.drop('Exited', axis=1)
y = df['Exited']

X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open('label_encoder_gender.pkl', 'wb') as file:
  pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
  pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
  pickle.dump(scaler, file)

In [ ]:
def create_model(neurons=32, layer=1, input_shape=None):
  model = Sequential()
  model.add(Dense(neurons, activation='relu', input_shape=input_shape))

  for _ in range (layer -1):
    model.add(Dense(neurons, activation='relu'))

  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [ ]:
# create a Keras Classifier

# Initialize with default training parameters AND parameters for create_model.
# These will be overridden by GridSearchCV's param_grid.
# Explicitly pass input_shape to KerasClassifier for the build_fn.
model = KerasClassifier(build_fn=create_model,
                        verbose=1,
                        epochs=50, # Default for KerasClassifier
                        batch_size=32, # Default for KerasClassifier
                        neurons=32, # Default for create_model
                        layer=1, # Default for create_model
                        input_shape=(X_train.shape[1],) # Crucial: Pass the input shape
                       )

In [ ]:
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layer': [1, 2],
    'epochs': [50, 100]
}

In [ ]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=3, verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

AttributeError: 'super' object has no attribute '__sklearn_tags__'